# Fraud Shield AI — EDA

This notebook is the visualization layer for the production code in `src/eda.py` and `src/features.py`. It profiles only `fraudTrain.csv`; the supplied `fraudTest.csv` remains a sealed, out-of-time holdout. Personally identifying fields (names, street addresses, full card numbers, and transaction IDs) are never retained in plot samples or displayed.

## 1. Reproducible configuration

The source CSV is scanned in bounded chunks. Exact aggregates use every training row. Correlation and distribution plots use deterministic hash samples, and velocity features are calculated on the complete ordered stream **before** sampling.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:  # Allows headless validation outside Jupyter.
    def display(value):
        print(value)

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'data' / 'fraudTrain.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from FraudShield/ or FraudShield/notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda import scan_fraud_csv, sha256_file

DATA_PATH = PROJECT_ROOT / 'data' / 'fraudTrain.csv'
SEED = 42
CHUNKSIZE = 100_000
CORRELATION_SAMPLE_ROWS = 75_000
LEGITIMATE_PLOT_ROWS = 50_000
FRAUD_PLOT_ROWS = 20_000

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 40)
print(f'Project root: {PROJECT_ROOT}')
print(f'Training source: {DATA_PATH} ({DATA_PATH.stat().st_size / 2**20:,.2f} MiB)')

In [ ]:
profile = scan_fraud_csv(
    DATA_PATH,
    chunksize=CHUNKSIZE,
    correlation_sample_size=CORRELATION_SAMPLE_ROWS,
    legitimate_plot_sample_size=LEGITIMATE_PLOT_ROWS,
    fraud_plot_sample_size=FRAUD_PLOT_ROWS,
    random_seed=SEED,
)
source_sha256 = sha256_file(DATA_PATH)
print(f'Profiled {profile.summary["rows"]:,} rows; SHA-256 {source_sha256}')

## 2. Data contract and quality

Semantic roles: `cc_num`, `zip`, and `trans_num` are opaque identifiers; `lat/long` are cardholder-home coordinates; `merch_lat/merch_long` are transaction merchant coordinates. The cosmetic `fraud_` prefix on every merchant name is not a label signal. The redundant unnamed CSV export index is validated and dropped.

In [ ]:
summary_keys = [
    'rows', 'display_time_start', 'display_time_end', 'amount_min', 'amount_max',
    'amount_mean', 'unique_cards', 'unique_merchants', 'unique_categories',
    'unique_states', 'velocity_clock', 'correlation_sample_rows',
    'visualization_sample_rows', 'sampling_seed',
]
display(pd.Series({key: profile.summary[key] for key in summary_keys}, name='value').to_frame())
display(profile.quality)
display(profile.missingness.query('missing_count > 0'))
if profile.missingness['missing_count'].sum() == 0:
    print('Missingness check: PASS — no missing cells in the training CSV.')

The single readable-timestamp backstep is a known leap-day/year-remapping artifact. `unix_time` stays monotonic and is used only as a causal elapsed-time clock. Its displayed calendar year is shifted seven years earlier than `trans_date_trans_time`, so the two raw time fields must not be treated as independent model signals.

## 3. Exact class imbalance

The positive rate is also the expected PR-AUC of a random ranking baseline. This makes raw accuracy actively misleading: a classifier predicting every transaction as legitimate would appear highly accurate while detecting no fraud.

In [ ]:
class_table = pd.DataFrame({
    'class': ['Legitimate', 'Fraud'],
    'count': [profile.summary['nonfraud_rows'], profile.summary['fraud_rows']],
})
class_table['share'] = class_table['count'] / class_table['count'].sum()
display(class_table.style.format({'count': '{:,.0f}', 'share': '{:.6%}'}))
print(f"Majority/minority ratio: {profile.summary['nonfraud_to_fraud_ratio']:.2f}:1")
print(f"Random-ranking PR-AUC baseline: {profile.summary['random_classifier_pr_auc']:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_table['class'], class_table['count'], color=['#4C78A8', '#E45756'])
ax.set_yscale('log')
ax.set_ylabel('Transactions (log scale)')
ax.set_title('Training class counts')
for bar, share in zip(bars, class_table['share']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.08, f'{share:.4%}', ha='center')
plt.tight_layout()
plt.show()

## 4. Correlation matrices

Pearson and Spearman correlations are shown on a uniform deterministic training sample. IDs and arbitrary ordinal encodings of categorical values are excluded. The rolling features below were computed causally on the full stream before the sample was selected.

In [ ]:
correlation_columns = [
    'log1p_amt', 'log1p_city_pop', 'distance_card_merchant_km', 'age_years',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'cc_txn_count_prev_1h', 'cc_amt_sum_prev_1h',
    'cc_txn_count_prev_6h', 'cc_amt_sum_prev_6h',
    'cc_txn_count_prev_24h', 'cc_amt_sum_prev_24h', 'is_fraud',
]
correlation_data = profile.correlation_sample[correlation_columns].dropna()
pearson = correlation_data.corr(method='pearson')
spearman = correlation_data.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for matrix, title, axis in zip(
    (pearson, spearman), ('Pearson correlation', 'Spearman correlation'), axes
):
    sns.heatmap(matrix, cmap='vlag', center=0, vmin=-1, vmax=1, ax=axis,
                square=True, cbar_kws={'shrink': 0.75})
    axis.set_title(title)
plt.tight_layout()
plt.show()

## 5. Fraud vs. legitimate amount and distance distributions

These ECDFs use all available sampled fraud rows plus a deterministic legitimate sample. Each line is normalized within class, so the comparison shows distribution shape rather than the overwhelming class-count difference.

In [ ]:
plot_sample = profile.visualization_sample.copy()
plot_sample['class'] = plot_sample['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.ecdfplot(data=plot_sample, x='log1p_amt', hue='class', ax=axes[0],
             palette={'Legitimate': '#4C78A8', 'Fraud': '#E45756'})
axes[0].set_xlabel('log(1 + transaction amount)')
axes[0].set_title('Amount distribution by class')
sns.ecdfplot(data=plot_sample, x='distance_card_merchant_km', hue='class', ax=axes[1],
             palette={'Legitimate': '#4C78A8', 'Fraud': '#E45756'})
axes[1].set_xlabel('Cardholder-to-merchant distance (km)')
axes[1].set_title('Geospatial distance distribution by class')
plt.tight_layout()
plt.show()

## 6. Temporal distributions

Weekly fraud rate answers a different question from within-class timing patterns, so both are retained. The rate plot uses exact aggregate counts; hour and weekday panels normalize within each class.

In [ ]:
weekly = profile.weekly_counts.pivot(index='week_start', columns='is_fraud', values='count').fillna(0)
weekly.columns = ['legitimate', 'fraud']
weekly['total'] = weekly.sum(axis=1)
weekly['fraud_rate'] = weekly['fraud'] / weekly['total']
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
axes[0].plot(weekly.index, weekly['total'], color='#4C78A8')
axes[0].set_ylabel('Transactions')
axes[0].set_title('Weekly transaction volume')
axes[1].plot(weekly.index, weekly['fraud_rate'] * 100, color='#E45756')
axes[1].axhline(profile.summary['fraud_rate'] * 100, color='black', linestyle='--', linewidth=1,
                label='Overall training rate')
axes[1].set_ylabel('Fraud rate (%)')
axes[1].set_title('Weekly fraud rate')
axes[1].legend()
plt.tight_layout()
plt.show()

def within_class_share(frame, key):
    result = frame.copy()
    result['class'] = result['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})
    result['share'] = result['count'] / result.groupby('class')['count'].transform('sum')
    return result

hourly = within_class_share(profile.hourly_counts, 'hour')
weekday = within_class_share(profile.weekday_counts, 'day_of_week')
weekday_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=hourly, x='hour', y='share', hue='class', marker='o', ax=axes[0],
             palette={'Legitimate': '#4C78A8', 'Fraud': '#E45756'})
axes[0].set_title('Within-class distribution by hour')
axes[0].set_ylabel('Share within class')
sns.lineplot(data=weekday, x='day_of_week', y='share', hue='class', marker='o', ax=axes[1],
             palette={'Legitimate': '#4C78A8', 'Fraud': '#E45756'})
axes[1].set_xticks(range(7), weekday_labels)
axes[1].set_title('Within-class distribution by weekday')
axes[1].set_ylabel('Share within class')
plt.tight_layout()
plt.show()

## 7. Geographic distributions

Merchant locations are aggregated into one-degree bins rather than plotting 1.3 million points. Fraud-rate bins with fewer than 250 transactions are suppressed to reduce unstable small-denominator patterns. `state` is the cardholder's home state, so it is reported separately from merchant geography.

In [ ]:
geo = profile.geographic_counts.pivot_table(
    index=['merchant_lat_bin', 'merchant_lon_bin'], columns='is_fraud',
    values='count', fill_value=0, aggfunc='sum'
).reset_index()
geo = geo.rename(columns={0: 'legitimate', 1: 'fraud'})
for column in ('legitimate', 'fraud'):
    if column not in geo:
        geo[column] = 0
geo['total'] = geo['legitimate'] + geo['fraud']
geo['fraud_rate'] = geo['fraud'] / geo['total']
supported_geo = geo.query('total >= 250')

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
density = axes[0].scatter(geo['merchant_lon_bin'], geo['merchant_lat_bin'],
                          c=np.log10(geo['total']), s=18, cmap='viridis')
axes[0].set_title('Merchant transaction density (log10 count)')
fig.colorbar(density, ax=axes[0], label='log10 transactions')
rate = axes[1].scatter(supported_geo['merchant_lon_bin'], supported_geo['merchant_lat_bin'],
                       c=supported_geo['fraud_rate'] * 100, s=22, cmap='magma')
axes[1].set_title('Merchant-bin fraud rate (support ≥ 250)')
fig.colorbar(rate, ax=axes[1], label='Fraud rate (%)')
for axis in axes:
    axis.set_xlabel('Merchant longitude')
    axis.set_ylabel('Merchant latitude')
plt.tight_layout()
plt.show()

state = profile.state_counts.pivot(index='state', columns='is_fraud', values='count').fillna(0)
state.columns = ['legitimate', 'fraud']
state['total'] = state.sum(axis=1)
state['fraud_rate'] = state['fraud'] / state['total']
state_supported = state.query('total >= 5000').nlargest(15, 'fraud_rate').sort_values('fraud_rate')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(state_supported.index, state_supported['fraud_rate'] * 100, color='#E45756')
ax.set_xlabel('Fraud rate (%)')
ax.set_title('Highest home-state fraud rates (support ≥ 5,000)')
plt.tight_layout()
plt.show()

## 8. Milestone 1 findings and handoff

- Fraud is extremely rare, so later evaluation must emphasize fraud recall, precision, PR-AUC, calibrated probabilities, and threshold/cost trade-offs rather than accuracy.
- The train/test pair is a continuous temporal stream. Milestone 2 should preserve chronology and keep all resampling strictly inside the training partition.
- Velocity state must flow forward into validation/test data without allowing labels or future transactions to flow backward. New cards receive explicit cold-start zeros.
- Display time is suitable for calendar plots, while monotonic `unix_time` is the only safe elapsed-time clock for rolling features.
- Geography represents cardholder-home versus merchant-location distance, not the physical path of a transaction.
- Raw PII and identifier values should remain excluded from reports, logs, and model explanations.

In [ ]:
run_manifest = pd.Series({
    'source': str(DATA_PATH),
    'sha256': source_sha256,
    'rows_profiled': profile.summary['rows'],
    'chunksize': CHUNKSIZE,
    'seed': SEED,
    'correlation_sample_rows': len(profile.correlation_sample),
    'visualization_sample_rows': len(profile.visualization_sample),
    'causal_clock': profile.summary['velocity_clock'],
}, name='run_manifest')
display(run_manifest.to_frame())